In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
df_metric = pd.read_csv('metric.csv')
df_log = pd.read_csv('log.csv')
df_trace = pd.read_csv('trace.csv')
df_error = pd.read_csv('error_logs.csv')

# Parse timestamps to UTC datetimes (Unix seconds -> UTC)
for df in (df_metric, df_log, df_trace, df_error):
    if 'timestamp' in df.columns:
        df['timestamp_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Basic file info: rows and timestamp range
metric_info = pd.DataFrame({
    'rows': [len(df_metric)],
    'min_timestamp': [df_metric['timestamp_dt'].min() if not df_metric.empty else None],
    'max_timestamp': [df_metric['timestamp_dt'].max() if not df_metric.empty else None]
}, index=['metric.csv'])

log_info = pd.DataFrame({
    'rows': [len(df_log)],
    'min_timestamp': [df_log['timestamp_dt'].min() if not df_log.empty else None],
    'max_timestamp': [df_log['timestamp_dt'].max() if not df_log.empty else None]
}, index=['log.csv'])

trace_info = pd.DataFrame({
    'rows': [len(df_trace)],
    'min_timestamp': [df_trace['timestamp_dt'].min() if not df_trace.empty else None],
    'max_timestamp': [df_trace['timestamp_dt'].max() if not df_trace.empty else None]
}, index=['trace.csv'])

error_info = pd.DataFrame({
    'rows': [len(df_error)],
    'min_timestamp': [df_error['timestamp_dt'].min() if not df_error.empty else None],
    'max_timestamp': [df_error['timestamp_dt'].max() if not df_error.empty else None]
}, index=['error_logs.csv'])

# Ensure numeric 'value' columns where applicable
if 'value' in df_metric.columns:
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
if 'value' in df_log.columns:
    df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')
if 'value' in df_trace.columns:
    df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# Aggregation function for metrics/logs/traces
def aggregate_value(df, group_cols, value_col='value', top_n=50):
    group = df.groupby(group_cols)[value_col].agg(
        count='count',
        min='min',
        max='max',
        mean='mean',
        p50=lambda x: x.quantile(0.5),
        p95=lambda x: x.quantile(0.95),
        p99=lambda x: x.quantile(0.99)
    ).reset_index()
    # Sort by count desc, keep at most top_n rows
    group_sorted = group.sort_values('count', ascending=False).head(top_n)
    # Round numeric columns for compactness
    num_cols = ['min','max','mean','p50','p95','p99']
    for c in num_cols:
        if c in group_sorted.columns:
            group_sorted[c] = pd.to_numeric(group_sorted[c]).round(6)
    return group_sorted

# 1) metric.csv aggregation
metric_agg = aggregate_value(df_metric, ['cmdb_id','kpi_name'])

# 2) log.csv aggregation (group by cmdb_id and log_name)
log_agg = aggregate_value(df_log, ['cmdb_id','log_name'])

# 3) trace.csv aggregation (group by cmdb_id and trace_name)
trace_agg = aggregate_value(df_trace, ['cmdb_id','trace_name'])

# 4) error_logs.csv: distinct cmdb_id (up to 20) and earliest 5 rows
error_cmdbs = pd.DataFrame({'cmdb_id': pd.unique(df_error['cmdb_id'])}).head(20)
error_head = df_error.sort_values('timestamp').head(5)[['timestamp_dt','cmdb_id','message']]

# Display compact results (variables returned)
metric_info, metric_agg, log_info, log_agg, trace_info, trace_agg, error_info, error_cmdbs, error_head

```
Out[1]:
```
summary = (
    "Summary of telemetry (UTC):\n\n"
    "metric.csv: 1849 rows. Timestamp range 2024-01-20 09:31:00+00:00 to 2024-01-20 09:55:00+00:00.\n"
    "- Aggregated by cmdb_id and kpi_name (top groups by count):\n"
    "  * carts / cpu (count 25) — min 1.680626, max 10.98546, mean 3.626423, p50 3.162373, p95 6.418995, p99 9.909564.\n"
    "  * carts / latency-50 and latency-90 present with small latency values (ms-scale fractions).\n"
    "  * Many services report large heap/memory KPI values (~2.1e8) across samples.\n"
    "  * Notable outlier: rabbitmq / diskio (count 25) with max 37246 and p99 28307 — indicates a large disk I/O spike on rabbitmq.\n\n"
    "log.csv: 456 rows. Timestamp range 2024-01-20 09:31:00+00:00 to 2024-01-20 09:55:00+00:00.\n"
    "- Top log groups (by count):\n"
    "  * front-end / log.row_count (count 25) — mean ~1885 rows per minute; front-end / log.error_count is 0 for these samples.\n"
    "  * catalogue and other services have row_count entries; most log.error_count groups are 0 (no errors logged in these aggregates).\n\n"
    "trace.csv: 0 rows (no trace telemetry available).\n\n"
    "error_logs.csv: 0 rows (no raw error log entries).\n\n"
    "Interpretation and next steps:\n"
    "- There are no error log or trace records to explain failures, and logs show error_count=0 for top services.\n"
    "- The clearest telemetry anomaly is a large disk I/O spike on rabbitmq (very high max and p99). This is the most likely lead to investigate first (disk I/O on rabbitmq). Check RabbitMQ host disk metrics, I/O wait, and queue/backlog metrics.\n"
    "- Also review memory and CPU KPIs for services showing high p99 values if investigation of RabbitMQ I/O does not explain the issue.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(            rows             min_timestamp             max_timestamp
metric.csv  1849 2024-01-20 09:31:00+00:00 2024-01-20 09:55:00+00:00,          cmdb_id    kpi_name  count           min           max          mean           p50           p95           p99
0          carts         cpu     25  1.680626e+00  1.098546e+01  3.626423e+00  3.162373e+00  6.418995e+00  9.909564e+00
1          carts  latency-50     25  9.476000e-03  1.715500e-02  1.368600e-02  1.416100e-02  1.697700e-02  1.711500e-02
2          carts  latency-90     25  2.185700e-02  1.352360e-01  7.338800e-02  3.059700e-02  1.325600e-01  1.346280e-01
3          carts         mem     25  1.990750e+08  2.142597e+08  2.103532e+08  2.119258e+08  2.141030e+08  2.142447e+08
4          carts      socket     25  8.266667e+00  1.100000e+01  9.706000e+00  9.900000e+00  1.094667e+01  1.100000e+01
..           ...         ...    ...           ...           ...           ...           ...           ...           ...
45  queue-master         mem     25  2.513111e+08  2.544533e+08  2.529518e+08  2.528317e+08  2.539925e+08  2.543677e+08
46  queue-master      socket     25  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00
47      rabbitmq         cpu     25  7.542720e-01  9.280210e-01  8.118000e-01  8.076980e-01  8.521590e-01  9.102790e-01
48      rabbitmq      diskio     25  0.000000e+00  3.724648e+04  1.489859e+03  0.000000e+00  0.000000e+00  2.830732e+04
49      rabbitmq         mem     25  8.908181e+07  9.416090e+07  9.088590e+07  8.992509e+07  9.411740e+07  9.415742e+07

[50 rows x 9 columns],          rows             min_timestamp             max_timestamp
log.csv   456 2024-01-20 09:31:00+00:00 2024-01-20 09:55:00+00:00,        cmdb_id         log_name  count  min   max     mean     p50     p95      p99
7    front-end    log.row_count     25  603  2178  1885.08  1944.0  2118.6  2167.68
6    front-end  log.error_count     25    0     0     0.00     0.0     0.0     0.00
5    catalogue    log.row_count     25   50   173   155.60   162.0   171.6   172.76
4    catalogue  log.error_count     25    0     0     0.00     0.0     0.0     0.00
12     payment  log.error_count     25    0     0     0.00     0.0     0.0     0.00
..         ...              ...    ...  ...   ...      ...     ...     ...      ...
17  session-db    log.row_count      5    5     5     5.00     5.0     5.0     5.00
22     user-db  log.error_count      2    0     0     0.00     0.0     0.0     0.00
23     user-db    log.row_count      2    1     1     1.00     1.0     1.0     1.00
10   orders-db  log.error_count      1    0     0     0.00     0.0     0.0     0.00
11   orders-db    log.row_count      1    1     1     1.00     1.0     1.0     1.00

[24 rows x 9 columns],            rows min_timestamp max_timestamp
trace.csv     0          None          None, Empty DataFrame
Columns: [cmdb_id, trace_name, count, min, max, mean, p50, p95, p99]
Index: [],                 rows min_timestamp max_timestamp
error_logs.csv     0          None          None, Empty DataFrame
Columns: [cmdb_id]
Index: [], Empty DataFrame
Columns: [timestamp_dt, cmdb_id, message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

# Reuse previously loaded df_metric if available; otherwise load
try:
    df_metric
except NameError:
    import pandas as pd
    df_metric = pd.read_csv('metric.csv')

import pandas as pd

# Ensure timestamp parsed to UTC datetime (Unix seconds)
if 'timestamp_dt' not in df_metric.columns:
    df_metric['timestamp_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
else:
    # ensure dtype is datetime64[ns, UTC]
    df_metric['timestamp_dt'] = pd.to_datetime(df_metric['timestamp_dt'], utc=True)

# Ensure value is numeric
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# 1) Compute global statistics (using the full series BEFORE any time filtering)
global_stats = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(
        total_count='count',
        p5=lambda x: x.quantile(0.05),
        p95=lambda x: x.quantile(0.95)
    )
    .reset_index()
)

# 2) Define incident window in UTC and filter rows
window_start = pd.Timestamp('2024-01-20 09:28:19', tz='UTC')
window_end = pd.Timestamp('2024-01-20 09:58:19', tz='UTC')
mask = (df_metric['timestamp_dt'] >= window_start) & (df_metric['timestamp_dt'] <= window_end)
df_win = df_metric[mask].copy()

# Total number of metric rows inside the incident window
window_total = len(df_win)

# 3) Per-group window aggregates
win_agg = (
    df_win
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(
        in_window_count='count',
        max_value_in_window='max',
        min_value_in_window='min'
    )
    .reset_index()
)

# 4) Determine anomalies in-window using global p5/p95 thresholds
# Merge thresholds into in-window rows
if not df_win.empty:
    df_win_thresh = df_win.merge(global_stats, on=['cmdb_id', 'kpi_name'], how='left')
    df_win_thresh['is_anomaly'] = df_win_thresh['value'].ge(df_win_thresh['p95']) | df_win_thresh['value'].le(df_win_thresh['p5'])
    anomaly_agg = (
        df_win_thresh[df_win_thresh['is_anomaly']]
        .groupby(['cmdb_id', 'kpi_name'])
        .agg(
            anomaly_count=('is_anomaly', 'sum'),
            earliest_anomaly_timestamp=('timestamp_dt', 'min')
        )
        .reset_index()
    )
else:
    anomaly_agg = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp'])

# 5) Combine global stats, window aggregates, and anomaly info
result = (
    global_stats
    .merge(win_agg, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(anomaly_agg, on=['cmdb_id', 'kpi_name'], how='left')
)

# Fill NaNs for groups with no in-window data / no anomalies
result['in_window_count'] = result['in_window_count'].fillna(0).astype(int)
result['anomaly_count'] = result['anomaly_count'].fillna(0).astype(int)
# earliest_anomaly_timestamp: keep NaT -> show None later
result['earliest_anomaly_timestamp'] = pd.to_datetime(result['earliest_anomaly_timestamp']).dt.tz_convert('UTC')

# For compactness, round numeric thresholds
result['p5'] = result['p5'].round(6)
result['p95'] = result['p95'].round(6)
result['max_value_in_window'] = result['max_value_in_window'].where(result['in_window_count']>0)
result['min_value_in_window'] = result['min_value_in_window'].where(result['in_window_count']>0)

# Select and order columns as requested
result_df = (
    result[
        ['cmdb_id','kpi_name','total_count','p5','p95',
         'in_window_count','anomaly_count','earliest_anomaly_timestamp',
         'max_value_in_window','min_value_in_window']
    ]
    .sort_values(['anomaly_count','in_window_count'], ascending=[False, False])
    .head(20)
    .reset_index(drop=True)
)

# Convert earliest_anomaly_timestamp NaT to None for clearer display
result_df['earliest_anomaly_timestamp'] = result_df['earliest_anomaly_timestamp'].where(
    result_df['anomaly_count']>0, None
)

# Compact outputs: top 20 rows and total window count
result_df, window_total

```
Out[2]:
```
summary = (
    "Summary (plain English, UTC):\n\n"
    "- Total metric rows inside the incident window [2024-01-20 09:28:19, 2024-01-20 09:58:19]: 1849.\n\n"
    "- Top affected (cmdb_id / kpi_name) by anomaly_count (shown in order):\n"
    "  1) catalogue / socket — 25 anomalies (all 25 in-window points). Earliest anomaly: 2024-01-20 09:31:00.\n"
    "  2) catalogue-db / diskio — 25/25 anomalous (all points). Earliest: 2024-01-20 09:31:00.\n"
    "  3) catalogue-db / socket — 25/25 anomalous. Earliest: 2024-01-20 09:31:00.\n"
    "  4) orders / diskio — 25/25 anomalous. Earliest: 2024-01-20 09:31:00.\n"
    "  5) orders-db / socket — 25/25 anomalous. Earliest: 2024-01-20 09:31:00.\n"
    "  6) queue-master / socket — 25/25 anomalous. Earliest: 2024-01-20 09:31:00.\n"
    "  7) rabbitmq / diskio — 25/25 anomalous; max in-window ≈ 37246 (very large I/O spike). Earliest: 2024-01-20 09:31:00.\n"
    "  8) rabbitmq / socket and rabbitmq-exporter (mem/socket) — fully anomalous in-window as well.\n"
    "  9) session-db / diskio — 19 anomalies out of 24 in-window points (earliest anomaly 2024-01-20 09:32:00).\n"
    " 10) user / socket and user-db / socket — 9 anomalies each (out of 25 in-window points).\n"
    " 11) carts (cpu, latency, mem) and payment/socket also show smaller anomaly counts (4 and 24 respectively for some KPIs).\n\n"
    "- Pattern and interpretation:\n"
    "  * Many socket and diskio KPIs show values outside their global P5–P95 thresholds for the entire sampled window (anomaly_count == in_window_count for many groups). That indicates a sustained deviation starting at ~2024-01-20 09:31:00 UTC for most of these groups.\n"
    "  * The most striking single anomaly is rabbitmq.diskio: global p95 was 0 but in-window max reached ~37246, indicating a major disk I/O spike on RabbitMQ.\n"
    "  * session-db.diskio also shows substantial I/O high values (max ~19523) with many anomalies.\n"
    "  * Widespread socket anomalies across catalogue, orders, queue-master, rabbitmq, and others suggest network/socket resource issues or connection saturation affecting multiple services.\n\n"
    "- Recommended next steps:\n"
    "  1) Immediately investigate RabbitMQ host(s) disk I/O, I/O wait, and queues/backlogs (highest-severity anomaly).\n"
    "  2) Check socket/connection counts and limits on catalogue, orders, queue-master, user, and related services (many socket KPIs are anomalous).\n"
    "  3) Inspect session-db disk I/O and database health (high disk activity observed).\n"
    "  4) Correlate with infrastructure metrics (disk I/O, IOPS, iowait, network) and application logs around 09:31 UTC to find root cause.\n\n"
    "Conclusion: The telemetry points to a major RabbitMQ disk I/O spike and widespread socket anomalies across multiple services starting ~09:31 UTC; RabbitMQ disk I/O is the highest-priority lead to investigate first."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name  total_count            p5           p95  in_window_count  anomaly_count earliest_anomaly_timestamp  max_value_in_window  min_value_in_window
0           catalogue      socket           25  5.306667e+00  6.000000e+00               25             25  2024-01-20 09:31:00+00:00         6.000000e+00         4.439024e+00
1        catalogue-db      diskio           25  0.000000e+00  0.000000e+00               25             25  2024-01-20 09:31:00+00:00         0.000000e+00         0.000000e+00
2        catalogue-db      socket           25  4.000000e+00  4.000000e+00               25             25  2024-01-20 09:31:00+00:00         4.000000e+00         3.317073e+00
3              orders      diskio           25  0.000000e+00  0.000000e+00               25             25  2024-01-20 09:31:00+00:00         0.000000e+00         0.000000e+00
4           orders-db      socket           25  5.000000e+00  6.000000e+00               25             25  2024-01-20 09:31:00+00:00         6.000000e+00         5.000000e+00
5        queue-master      socket           25  3.000000e+00  3.000000e+00               25             25  2024-01-20 09:31:00+00:00         3.000000e+00         3.000000e+00
6            rabbitmq      diskio           25  0.000000e+00  0.000000e+00               25             25  2024-01-20 09:31:00+00:00         3.724648e+04         0.000000e+00
7            rabbitmq      socket           25  1.100000e+01  1.100000e+01               25             25  2024-01-20 09:31:00+00:00         1.100000e+01         1.100000e+01
8   rabbitmq-exporter         mem           25  4.689920e+06  4.689920e+06               25             25  2024-01-20 09:31:00+00:00         4.689920e+06         4.689920e+06
9   rabbitmq-exporter      socket           25  1.000000e+00  1.000000e+00               25             25  2024-01-20 09:31:00+00:00         1.000000e+00         1.000000e+00
10         session-db      socket           25  3.000000e+00  3.000000e+00               25             25  2024-01-20 09:31:00+00:00         3.000000e+00         3.000000e+00
11           carts-db      socket           25  5.000000e+00  6.000000e+00               25             24  2024-01-20 09:31:00+00:00         6.000000e+00         5.000000e+00
12            payment      socket           25  3.000000e+00  4.000000e+00               25             24  2024-01-20 09:31:00+00:00         4.000000e+00         3.000000e+00
13         session-db      diskio           24  0.000000e+00  1.290917e+04               24             19  2024-01-20 09:32:00+00:00         1.952393e+04         0.000000e+00
14               user      socket           25  1.216667e+01  1.700000e+01               25              9  2024-01-20 09:31:00+00:00         1.700000e+01         1.012195e+01
15            user-db      socket           25  6.213333e+00  8.000000e+00               25              9  2024-01-20 09:31:00+00:00         8.000000e+00         6.000000e+00
16              carts         cpu           25  1.716344e+00  6.418995e+00               25              4  2024-01-20 09:31:00+00:00         1.098546e+01         1.680626e+00
17              carts  latency-50           25  9.555000e-03  1.697700e-02               25              4  2024-01-20 09:31:00+00:00         1.715530e-02         9.476278e-03
18              carts  latency-90           25  2.192600e-02  1.325600e-01               25              4  2024-01-20 09:39:00+00:00         1.352361e-01         2.185738e-02
19              carts         mem           25  2.018405e+08  2.141030e+08               25              4  2024-01-20 09:31:00+00:00         2.142597e+08         1.990750e+08, 1849)```
```